# Feature Engineering Study - Tabular (Regressao)Estudo comparativo de tecnicas de feature engineering para regressao no dataset**California Housing** (20.640 amostras, 8 features, target = preco mediano em$100.000). Cada tecnica e avaliada com 3 modelos: **LinearRegression** (sensivela escala), **LightGBM** (gradient boosting) e **RandomForest**.Pergunta central: *quanto feature engineering ajuda cada tipo de modelo?*

In [1]:
import numpy as npimport pandas as pdimport timeimport warningswarnings.filterwarnings('ignore')from sklearn.datasets import fetch_california_housingfrom sklearn.model_selection import train_test_splitfrom sklearn.preprocessing import StandardScaler, MinMaxScaler, PolynomialFeatures, KBinsDiscretizerfrom sklearn.decomposition import PCAfrom sklearn.linear_model import LinearRegressionfrom sklearn.ensemble import RandomForestRegressorfrom sklearn.metrics import mean_absolute_error, r2_scoreimport lightgbm as lgbSEED = 42np.random.seed(SEED)print('Imports OK')

Imports OK


## 1. Carregamento do Dataset

In [2]:
data = fetch_california_housing()X = pd.DataFrame(data.data, columns=data.feature_names)y = data.targetprint(f'Shape: {X.shape}')X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)print(f'Train: {len(X_train)}, Test: {len(X_test)}')

Shape: (20640, 8)
Train: 16512, Test: 4128


## 2. Tecnicas de Feature Engineering

| # | Tecnica | Descricao ||---|---------|------------|| 1 | Raw | Features originais || 2 | Standardized | StandardScaler || 3 | MinMax | MinMaxScaler || 4 | Polynomial (d=2) | 44 features || 5 | Poly interactions | 36 features (sem x^2) || 6 | Log transform | log1p em features assimetricas || 7 | Binning | Discretizacao em 10 bins || 8 | PCA (95%) | Reducao para 6 features || 9 | Geo features | distancia ao centro + razoes - 12 features || 10 | Combined | Log + Geo + Poly + Standard - 55 features |

In [3]:
# Funcoes de feature engineering (ver _run_fe_tabular.py para codigo completo)# Cada funcao recebe X_train, X_test e retorna as matrizes transformadas.TECHNIQUES = [    ('1. Raw', fe_raw),    ('2. Standardized', fe_standardized),    ('3. MinMax', fe_minmax),    ('4. Polynomial (d=2)', fe_polynomial),    ('5. Poly interactions', fe_poly_interactions),    ('6. Log transform', fe_log_transform),    ('7. Binning (10 bins)', fe_binning),    ('8. PCA (95% var)', fe_pca),    ('9. Geo features', fe_geo),    ('10. Combined', fe_combined),]

## 3. Resultados

In [4]:
# Resultados da execucao completa (script _run_fe_tabular.py)# Cada tecnica x 3 modelos = 30 experimentos

Technique                           Model              R2       MAE      Features   Time
------------------------------------------------------------------------------------------
1. Raw                              LinearRegression   0.5758   0.5332   8          0.00
1. Raw                              LightGBM           0.8360   0.3078   8          0.33
1. Raw                              RandomForest       0.8051   0.3275   8          7.36
2. Standardized                     LinearRegression   0.5758   0.5332   8          0.03
2. Standardized                     LightGBM           0.8386   0.3055   8          3.57
2. Standardized                     RandomForest       0.8053   0.3274   8          16.20
3. MinMax                           LinearRegression   0.5758   0.5332   8          0.04
3. MinMax                           LightGBM           0.8365   0.3079   8          1.54
3. MinMax                           RandomForest       0.8044   0.3275   8          12.21
4. Polynomial (d=

## 4. Tabela Resumo (R^2 por modelo)| Tecnica | LinearRegression | LightGBM | RandomForest ||---------|------------------|----------|-------------|| 1. Raw | 0,5758 | 0,8360 | 0,8051 || 2. Standardized | 0,5758 | 0,8386 | 0,8053 || 3. MinMax | 0,5758 | 0,8365 | 0,8044 || **4. Polynomial (d=2)** | **0,6457** | 0,8346 | 0,7968 || 5. Poly interactions | 0,6225 | 0,8339 | 0,7969 || 6. Log transform | 0,6114 | 0,8360 | 0,8053 || 7. Binning | 0,5858 | 0,8288 | **0,8154** || 8. PCA (95%) | 0,4877 | 0,6583 | 0,6422 || **9. Geo features** | 0,5945 | **0,8418** | **0,8205** || **10. Combined** | **0,7112** | 0,8375 | 0,8045 |

## 5. Analise### 1. LinearRegression - Feature Engineering e Crucial (+13,5 pp)Modelos lineares sao fortemente beneficiados: combined (log + poly + geo + standard)alcanca R^2 = 0,7112 - um ganho de **+13,5 pp** sobre o baseline (0,5758). Polynomialfeatures sozinho ja da +7,0 pp, permitindo capturar relacoes nao-lineares(ex: MedInc x Latitude) sem trocar de modelo. Log transform tambem ajuda (+3,6 pp)ao reduzir o impacto de outliers em features com distribuicao assimetrica(AveRooms, Population, AveOccup).### 2. LightGBM - Pouco Beneficiado (+0,6 pp)Gradient boosting trees sao invariantes a escala e capturam nao-linearidadesnativamente via splits. A unica tecnica que ajuda e **Geo features** (+0,6 pp),pois cria features de domain knowledge que as arvores nao conseguem inferirautomaticamente (ex: distancia euclidiana ao centro nao e uma funcao desplits univariados). Todas as outras tecnicas (scaling, polynomial, PCA) naotrazem ganho relevante.### 3. RandomForest - Beneficio Intermediario (+1,5 pp)RandomForest se beneficia de **Geo features** (+1,5 pp) pelo mesmo motivo queLightGBM. **Binning** tambem ajuda (+1,0 pp): discretizacao em 10 bins facilitaos splits nas arvores, que tem profundidade limitada e nao conseguem capturarbordas finas em features continuas.### 4. PCA - Prejudicial para TodosPCA perdeu 8,8 a 17,8 pp em todos os modelos. O problema e que PCA reduz adimensionalidade projetando para componentes que maximizam a variancia - masvariancia nao e sinonimo de poder preditivo. Neste dataset, Latitude e Longitudesao colineares, e PCA funde-as em um componente que perde a informacaodirecional (norte-sul) crucial para precos imobiliarios.### Conclusoes1. **O valor do feature engineering depende do modelo**: LinearRegression ganhou   13,5 pp; LightGBM apenas 0,6 pp. O esforco deve ser proporcional a sensibilidade   do modelo.2. **Polynomial features e a tecnica mais impactante para modelos lineares**   (+7,0 pp sozinho), permitindo capturar nao-linearidades sem trocar de modelo.3. **Arvores so se beneficiam de features de domain knowledge** (Geo features),   nao de transformadas matematicas (scaling, PCA).4. **PCA deve ser evitado** - aqui perdeu 9-18 pp em todos os modelos.5. **Recomendacao**: para modelos lineares, Combined (log + poly + geo + standard)   e otimo. Para trees, Raw + Geo features ja e suficiente.